# 제4장: AI 거버넌스 조직 체계

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/back2zion/ai-governance-handbook/blob/main/notebooks/ch04.ipynb)

> **『AI 거버넌스 실무 지침서』** (곽두일) 실습 코드
>
> 이 노트북의 모든 코드는 Google Colab에서 바로 실행할 수 있습니다.

In [ ]:
# 필요 패키지 설치 (최초 1회)
!pip install -q fairlearn shap mlflow diffprivlib alibi alibi-detect evidently pandas scikit-learn matplotlib seaborn numpy

### AI 거버넌스 조직 체계 구축

## AI 거버넌스 조직 Model

### 거버넌스 조직의 핵심 기능

### 조직 모델 유형

### 조직 규모별 권장 Model

## Role과 책임 Definition

### 핵심 역할 Definition

### RACI 매트릭스: AI 생명주기 활동별

### 역할 정의서 템플릿

**Automated Role Definition Generation**

In [ ]:
from dataclasses import dataclass, field
from typing import List, Optional

@dataclass
class RoleDefinition:
    """AI Governance Role Definition Template"""
    role_title: str
    department: str
    reports_to: str
    purpose: str
    responsibilities: List[str] = field(default_factory=list)
    kpis: List[str] = field(default_factory=list)

    def to_summary(self) -> str:
        resps = "\n".join([f"- {r}" for r in self.responsibilities])
        return f"Role: {self.role_title}\nDept: {self.department}\n\nPurpose:\n{self.purpose}\n\nTasks:\n{resps}"

# Example Usage
caio = RoleDefinition(
    role_title="Chief AI Officer (CAIO)",
    department="Executive",
    reports_to="CEO",
    purpose="Lead 전사 AI 거버넌스 및 전략 통합",
    responsibilities=["정책 승인", "고위험 AI 감독", "규제 대응"]
)

## 거버넌스 프로세스 설계

### AI 시스템 생명주기 거버넌스

### 리스크 based 거버넌스 적용

### 배포 승인 프로세스

**Automated Deployment Approval System**

In [ ]:
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Any
from datetime import datetime
from enum import Enum
import uuid

class ApprovalStatus(Enum):
 """Approval Status"""
 PENDING = "pending"
 DOCUMENT_REVIEW = "document_review"
 TECHNICAL_REVIEW = "technical_review"
 GOVERNANCE_REVIEW = "governance_review"
 COMMITTEE_REVIEW = "committee_review"
 APPROVED = "approved"
 REJECTED = "rejected"
 REVISION_REQUIRED = "revision_required"

class RiskLevel(Enum):
 """ Level"""
 HIGH = "high"
 MEDIUM = "medium"
 LOW = "low"

@dataclass
class ApprovalRequest:
 """ Approval """
 
 request_id: str = field(
 default_factory=lambda: f"APR-{datetime.now().strftime('%Y%m%d')}-{uuid.uuid4().hex[:6].upper()}"
 )
 system_id: str = ""
 system_name: str = ""
 version: str = ""
 requester: str = ""
 request_date: datetime = field(default_factory=datetime.now)
 
 risk_level: RiskLevel = RiskLevel.MEDIUM
 high_risk_sector: Optional[str] = None
 
 status: ApprovalStatus = ApprovalStatus.PENDING
 
 # Checklist 
 document_checklist: Dict[str, bool] = field(default_factory=dict)
 technical_checklist: Dict[str, bool] = field(default_factory=dict)
 fairness_results: Dict[str, float] = field(default_factory=dict)
 
 # Approval 
 approval_history: List[Dict[str, Any]] = field(default_factory=list)
 
 # Final 
 approved_by: str = ""
 approved_date: Optional[datetime] = None
 rejection_reason: str = ""
 conditions: List[str] = field(default_factory=list)


class DeploymentApprovalSystem:
 """
 AI System Approval Management System
 
 15 Governance Approval Operation
 """
 
 # Document Checklist
 REQUIRED_DOCUMENTS = {
 RiskLevel.HIGH: [
 "ai_system_registration",
 "training_data_spec",
 "model_card",
 "fairness_report",
 "risk_assessment",
 "human_oversight_plan",
 "privacy_impact_assessment",
 "test_results",
 "monitoring_plan",
 ],
 RiskLevel.MEDIUM: [
 "ai_system_registration",
 "model_card",
 "fairness_report",
 "test_results",
 "monitoring_plan",
 ],
 RiskLevel.LOW: [
 "ai_system_registration",
 "simplified_model_card",
 "test_results",
 ],
 }
 
 # TECHNICAL_CRITERIA = {
 'min_accuracy': 0.8,
 'min_fairness_dp_ratio': 0.8,
 'max_fairness_eo_diff': 0.1,
 'min_robustness_score': 0.9,
 }
 
 def __init__(self):
 self.requests: Dict[str, ApprovalRequest] = {}
 
 def submit_request(
 self,
 system_id: str,
 system_name: str,
 version: str,
 requester: str,
 risk_level: RiskLevel,
 high_risk_sector: Optional[str] = None,
 ) -> ApprovalRequest:
 """ Approval """
 
 request = ApprovalRequest(
 system_id=system_id,
 system_name=system_name,
 version=version,
 requester=requester,
 risk_level=risk_level,
 high_risk_sector=high_risk_sector,
 )
 
 self.requests[request.request_id] = request
 self._add_history(request, "REQUEST_SUBMITTED", requester)
 
 return request
 
 def review_documents(
 self,
 request_id: str,
 checklist_results: Dict[str, bool],
 reviewer: str,
 ) -> tuple:
 """Document """
 
 request = self.requests.get(request_id)
 if not request:
 raise ValueError(f" : {request_id}")
 
 request.document_checklist = checklist_results
 request.status = ApprovalStatus.DOCUMENT_REVIEW
 
 # Document 
 required = self.REQUIRED_DOCUMENTS[request.risk_level]
 missing = [doc for doc in required if not checklist_results.get(doc, False)]
 
 passed = len(missing) == 0
 
 self._add_history(
 request,
 "DOCUMENT_REVIEW_COMPLETE" if passed else "DOCUMENT_REVIEW_FAILED",
 reviewer,
 {"missing_documents": missing}
 )
 
 if not passed:
 request.status = ApprovalStatus.REVISION_REQUIRED
 
 return passed, missing
 
 def review_technical(
 self,
 request_id: str,
 accuracy: float,
 fairness_dp_ratio: float,
 fairness_eo_diff: float,
 robustness_score: float,
 reviewer: str,
 ) -> tuple:
 """ """
 
 request = self.requests.get(request_id)
 if not request:
 raise ValueError(f" : {request_id}")
 
 request.technical_checklist = {
 'accuracy': accuracy,
 'fairness_dp_ratio': fairness_dp_ratio,
 'fairness_eo_diff': fairness_eo_diff,
 'robustness_score': robustness_score,
 }
 
 request.fairness_results = {
 'dp_ratio': fairness_dp_ratio,
 'eo_diff': fairness_eo_diff,
 }
 
 request.status = ApprovalStatus.TECHNICAL_REVIEW
 
 # failures = []
 
 if accuracy < self.TECHNICAL_CRITERIA['min_accuracy']:
 failures.append(f"Accuracy : {accuracy:.3f} < {self.TECHNICAL_CRITERIA['min_accuracy']}")
 
 if fairness_dp_ratio < self.TECHNICAL_CRITERIA['min_fairness_dp_ratio']:
 failures.append(f"Fairness(DP Ratio) : {fairness_dp_ratio:.3f} < {self.TECHNICAL_CRITERIA['min_fairness_dp_ratio']}")
 
 if fairness_eo_diff > self.TECHNICAL_CRITERIA['max_fairness_eo_diff']:
 failures.append(f"Fairness(EO Diff) : {fairness_eo_diff:.3f} > {self.TECHNICAL_CRITERIA['max_fairness_eo_diff']}")
 
 if robustness_score < self.TECHNICAL_CRITERIA['min_robustness_score']:
 failures.append(f" : {robustness_score:.3f} < {self.TECHNICAL_CRITERIA['min_robustness_score']}")
 
 passed = len(failures) == 0
 
 self._add_history(
 request,
 "TECHNICAL_REVIEW_COMPLETE" if passed else "TECHNICAL_REVIEW_FAILED",
 reviewer,
 {"failures": failures}
 )
 
 if not passed:
 request.status = ApprovalStatus.REVISION_REQUIRED
 
 return passed, failures
 
 def approve(
 self,
 request_id: str,
 approver: str,
 conditions: Optional[List[str]] = None,
 ) -> ApprovalRequest:
 """Final Approval"""
 
 request = self.requests.get(request_id)
 if not request:
 raise ValueError(f" : {request_id}")
 
 # System Approval 
 if request.risk_level == RiskLevel.HIGH:
 request.status = ApprovalStatus.COMMITTEE_REVIEW
 else:
 request.status = ApprovalStatus.GOVERNANCE_REVIEW
 
 request.approved_by = approver
 request.approved_date = datetime.now()
 request.conditions = conditions or []
 request.status = ApprovalStatus.APPROVED
 
 self._add_history(
 request,
 "APPROVED",
 approver,
 {"conditions": conditions}
 )
 
 return request
 
 def reject(
 self,
 request_id: str,
 rejector: str,
 reason: str,
 ) -> ApprovalRequest:
 """ """
 
 request = self.requests.get(request_id)
 if not request:
 raise ValueError(f" : {request_id}")
 
 request.status = ApprovalStatus.REJECTED
 request.rejection_reason = reason
 
 self._add_history(
 request,
 "REJECTED",
 rejector,
 {"reason": reason}
 )
 
 return request
 
 def _add_history(
 self,
 request: ApprovalRequest,
 action: str,
 actor: str,
 details: Optional[Dict] = None,
 ) -> None:
 """Approval Addition"""
 
 request.approval_history.append({
 'timestamp': datetime.now().isoformat(),
 'action': action,
 'actor': actor,
 'status': request.status.value,
 'details': details or {},
 })
 
 def generate_approval_certificate(
 self,
 request_id: str,
 ) -> str:
 """Approval Generation (LaTeX)"""
 
 request = self.requests.get(request_id)
 if not request or request.status != ApprovalStatus.APPROVED:
 raise ValueError("Approval ")
 
 cert = r"""
\begin{table}[htbp]
\centering
\caption{AI System Approval }
\begin{tabularx}{\textwidth}{|p{3.5cm}|X|}
\hline
\multicolumn{2}{|c|}{\textbf{AI System Approval }} \\
\hline
\multicolumn{2}{|c|}{\textit{Certificate of AI System Deployment Approval}} \\
\hline
Approval & """ + request.request_id + r""" \\
\hline
System & """ + request.system_name + r""" \\
\hline
System ID & """ + request.system_id + r""" \\
\hline
Version & """ + request.version + r""" \\
\hline
 Level & """ + request.risk_level.value.upper() + r""" \\
\hline
"""
 
 if request.high_risk_sector:
 cert += r" & " + request.high_risk_sector + r" \\" + "\n" + r"\hline" + "\n"
 
 cert += r"""
Approval & """ + request.approved_by + r""" \\
\hline
Approval days & """ + request.approved_date.strftime('%Y-%m-%d') + r""" \\
\hline
"""
 
 if request.conditions:
 cert += r"""
\multicolumn{2}{|c|}{\textbf{Approval }} \\
\hline
\multicolumn{2}{|p{13cm}|}{
\begin{itemize}[nosep,leftmargin=*]
"""
 for cond in request.conditions:
 cert += r"\item " + cond + "\n"
 
 cert += r"""
\end{itemize}
} \\
\hline
"""
 
 cert += r"""
\multicolumn{2}{|c|}{\textbf{Fairness Assessment }} \\
\hline
DP Ratio & """ + f"{request.fairness_results.get('dp_ratio', 0):.4f}" + r""" \\
\hline
EO Difference & """ + f"{request.fairness_results.get('eo_diff', 0):.4f}" + r""" \\
\hline
\multicolumn{2}{|p{13.5cm}|}{
\small AI 15 5years .
} \\
\hline
\end{tabularx}
\end{table}
"""
 
 return cert

### 모니터링 및 지속적 개선

**AI System Monitoring Dashboard**

In [ ]:
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Callable
from datetime import datetime, timedelta
from enum import Enum
import numpy as np

class AlertSeverity(Enum):
 """ """
 INFO = "info"
 WARNING = "warning"
 CRITICAL = "critical"

@dataclass
class MonitoringAlert:
 """Monitoring """
 alert_id: str
 system_id: str
 metric_name: str
 current_value: float
 threshold: float
 severity: AlertSeverity
 timestamp: datetime
 message: str
 acknowledged: bool = False
 resolved: bool = False

@dataclass
class MonitoringConfig:
 """Monitoring Setup"""
 
 # Performance 
 accuracy_warning: float = 0.05 # 5% accuracy_critical: float = 0.10 # 10% # Fairness 
 dp_ratio_min: float = 0.8 # 4/5 
 eo_diff_max: float = 0.1
 
 # Drift 
 psi_warning: float = 0.1
 psi_critical: float = 0.2
 
 # Operation 
 latency_p99_ms: float = 500.0
 error_rate_max: float = 0.01 # 1%


class AISystemMonitor:
 """
 AI System Monitoring
 
 Performance, Fairness, Drift Tracking Generation
 """
 
 def __init__(
 self,
 system_id: str,
 system_name: str,
 config: Optional[MonitoringConfig] = None,
 ):
 self.system_id = system_id
 self.system_name = system_name
 self.config = config or MonitoringConfig()
 
 # ( )
 self.baseline_metrics: Dict[str, float] = {}
 
 # self.current_metrics: Dict[str, float] = {}
 
 # self.metric_history: Dict[str, List[tuple]] = {}
 
 # self.active_alerts: List[MonitoringAlert] = []
 
 # self.alert_callbacks: List[Callable] = []
 
 def set_baseline(self, metrics: Dict[str, float]) -> None:
 """ Setup"""
 self.baseline_metrics = metrics.copy()
 
 def register_alert_callback(self, callback: Callable) -> None:
 """ (Slack, days )"""
 self.alert_callbacks.append(callback)
 
 def record_metrics(
 self,
 metrics: Dict[str, float],
 timestamp: Optional[datetime] = None,
 ) -> List[MonitoringAlert]:
 """ """
 
 timestamp = timestamp or datetime.now()
 self.current_metrics = metrics.copy()
 
 # for name, value in metrics.items():
 if name not in self.metric_history:
 self.metric_history[name] = []
 self.metric_history[name].append((timestamp, value))
 
 # new_alerts = self._check_thresholds(metrics, timestamp)
 
 # for alert in new_alerts:
 for callback in self.alert_callbacks:
 try:
 callback(alert)
 except Exception as e:
 print(f"Alert callback failed: {e}")
 
 return new_alerts
 
 def _check_thresholds(
 self,
 metrics: Dict[str, float],
 timestamp: datetime,
 ) -> List[MonitoringAlert]:
 """ Generation"""
 
 alerts = []
 
 # Accuracy 
 if 'accuracy' in metrics and 'accuracy' in self.baseline_metrics:
 baseline = self.baseline_metrics['accuracy']
 current = metrics['accuracy']
 degradation = (baseline - current) / baseline
 
 if degradation >= self.config.accuracy_critical:
 alerts.append(self._create_alert(
 'accuracy', current, baseline,
 AlertSeverity.CRITICAL, timestamp,
 f"Accuracy : {degradation:.1%} "
 f"( : {baseline:.4f} -> : {current:.4f})"
 ))
 elif degradation >= self.config.accuracy_warning:
 alerts.append(self._create_alert(
 'accuracy', current, baseline,
 AlertSeverity.WARNING, timestamp,
 f"Accuracy : {degradation:.1%} "
 ))
 
 # Fairness (DP Ratio)
 if 'dp_ratio' in metrics:
 dp_ratio = metrics['dp_ratio']
 if dp_ratio < self.config.dp_ratio_min:
 alerts.append(self._create_alert(
 'dp_ratio', dp_ratio, self.config.dp_ratio_min,
 AlertSeverity.CRITICAL, timestamp,
 f"Fairness (4/5 ) : DP Ratio = {dp_ratio:.4f} "
 f"( : >= {self.config.dp_ratio_min})"
 ))
 
 # Fairness (EO Diff)
 if 'eo_diff' in metrics:
 eo_diff = metrics['eo_diff']
 if eo_diff > self.config.eo_diff_max:
 alerts.append(self._create_alert(
 'eo_diff', eo_diff, self.config.eo_diff_max,
 AlertSeverity.CRITICAL, timestamp,
 f"Fairness : EO Diff = {eo_diff:.4f} "
 f"( : <= {self.config.eo_diff_max})"
 ))
 
 # Data Drift (PSI)
 if 'psi' in metrics:
 psi = metrics['psi']
 if psi >= self.config.psi_critical:
 alerts.append(self._create_alert(
 'psi', psi, self.config.psi_critical,
 AlertSeverity.CRITICAL, timestamp,
 f" Data Drift : PSI = {psi:.4f}"
 ))
 elif psi >= self.config.psi_warning:
 alerts.append(self._create_alert(
 'psi', psi, self.config.psi_warning,
 AlertSeverity.WARNING, timestamp,
 f"Data Drift : PSI = {psi:.4f}"
 ))
 
 # if 'latency_p99_ms' in metrics:
 latency = metrics['latency_p99_ms']
 if latency > self.config.latency_p99_ms:
 alerts.append(self._create_alert(
 'latency_p99_ms', latency, self.config.latency_p99_ms,
 AlertSeverity.WARNING, timestamp,
 f" SLA : P99 = {latency:.0f}ms "
 f"( : <= {self.config.latency_p99_ms:.0f}ms)"
 ))
 
 # if 'error_rate' in metrics:
 error_rate = metrics['error_rate']
 if error_rate > self.config.error_rate_max:
 alerts.append(self._create_alert(
 'error_rate', error_rate, self.config.error_rate_max,
 AlertSeverity.CRITICAL, timestamp,
 f" : {error_rate:.2%} "
 f"( : <= {self.config.error_rate_max:.2%})"
 ))
 
 self.active_alerts.extend(alerts)
 return alerts
 
 def _create_alert(
 self,
 metric_name: str,
 current_value: float,
 threshold: float,
 severity: AlertSeverity,
 timestamp: datetime,
 message: str,
 ) -> MonitoringAlert:
 """ Generation"""
 
 import uuid
 
 return MonitoringAlert(
 alert_id=f"ALT-{uuid.uuid4().hex[:8].upper()}",
 system_id=self.system_id,
 metric_name=metric_name,
 current_value=current_value,
 threshold=threshold,
 severity=severity,
 timestamp=timestamp,
 message=message,
 )
 
 def calculate_psi(
 self,
 baseline_distribution: np.ndarray,
 current_distribution: np.ndarray,
 bins: int = 10,
 ) -> float:
 """
 Population Stability Index (PSI) Calculation
 
 PSI < 0.1: None
 0.1 <= PSI < 0.2: PSI >= 0.2: ( )
 """
 
 # days bin edge 
 min_val = min(baseline_distribution.min(), current_distribution.min())
 max_val = max(baseline_distribution.max(), current_distribution.max())
 bin_edges = np.linspace(min_val, max_val, bins + 1)
 
 # Calculation
 baseline_hist, _ = np.histogram(baseline_distribution, bins=bin_edges)
 current_hist, _ = np.histogram(current_distribution, bins=bin_edges)
 
 # Calculation (0 )
 baseline_pct = (baseline_hist + 1) / (len(baseline_distribution) + bins)
 current_pct = (current_hist + 1) / (len(current_distribution) + bins)
 
 # PSI Calculation
 psi = np.sum((current_pct - baseline_pct) * np.log(current_pct / baseline_pct))
 
 return psi
 
 def generate_daily_report(self, date: Optional[datetime] = None) -> str:
 """days Monitoring Report Generation"""
 
 date = date or datetime.now()
 date_str = date.strftime('%Y-%m-%d')
 
 report = f"""
# AI System days Monitoring Report

System: {self.system_name} ({self.system_id}) 
days: {date_str}

---

## Status 

| | | | | Status |
|--------|---------|--------|------|------|
"""
 
 for metric, current in self.current_metrics.items():
 baseline = self.baseline_metrics.get(metric, '-')
 if isinstance(baseline, float):
 change = ((current - baseline) / baseline) * 100
 change_str = f"{change:+.1f}%"
 if metric in ['accuracy', 'dp_ratio']:
 status = "[PASS]" if current >= baseline * 0.95 else "[WARN]"
 elif metric in ['eo_diff', 'psi', 'error_rate']:
 status = "[PASS]" if current <= baseline * 1.1 else "[WARN]"
 else:
 status = "-"
 else:
 change_str = "-"
 status = "-"
 
 report += f"| {metric} | {current:.4f} | {baseline if isinstance(baseline, str) else f'{baseline:.4f}'} | {change_str} | {status} |\n"
 
 # today_alerts = [
 a for a in self.active_alerts
 if a.timestamp.date() == date.date() and not a.resolved
 ]
 
 if today_alerts:
 report += f"""

---

## ({len(today_alerts)} )

| | | | |
|------|--------|--------|--------|
"""
 for alert in today_alerts:
 report += f"| {alert.timestamp.strftime('%H:%M')} | {alert.severity.value} | {alert.metric_name} | {alert.message} |\n"
 else:
 report += "\n\n---\n\n## \n\n None [PASS]\n"
 
 report += f"""

---

* Report Generation .*
* 15 5years .*
"""
 
 return report

## 글로벌 선도 기업의 AI 거버넌스 조직: 2026년 현황

### 빅테크의 AI 거버넌스 조직 구조

### 금융 및 공공부문의 AI 거버넌스 조직

### 한국 기업에의 시사점: 3가지 핵심 교훈

### 제4장 요약